# 03 · Train LSTNet-Skip on Google Colab

Open this notebook in Colab:
`https://colab.research.google.com/github/Chandana18G/lstnet-reproduction/blob/main/notebooks/03_train_on_colab.ipynb`

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells from top to bottom.

Checkpoints go to **Google Drive**, so a run that is cut off (Colab time limit, closed tab)
continues where it stopped: just run the training cell again with the same settings.

## 1 · Check that a GPU is there

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

## 2 · Mount Google Drive (for checkpoints and results)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/lstnet-checkpoints"

## 3 · Get the code and the data

In [ ]:
%cd /content
!git clone -q https://github.com/Chandana18G/lstnet-reproduction.git || (cd lstnet-reproduction && git pull -q)
%cd /content/lstnet-reproduction
!bash scripts/download_data.sh

## 4 · Train

Settings = the authors' `ele.sh` (horizon 24, linear output) with their defaults:
window 168, CNN 100, GRU 100, skip 24, skip-GRU 5, AR window 24, dropout 0.2, Adam 1e-3,
gradient clip 10, 100 epochs, L1 loss. Change `SEED` for the multi-seed runs (0 … 9).

In [ ]:
SEED = 0
!python src/train.py --data electricity --horizon 24 --skip 24 --seed {SEED} --checkpoint-dir {CKPT}

## 5 · Keep the result

In [ ]:
import json, shutil, glob
for f in glob.glob("results/runs/*/metrics.json"):
    r = json.load(open(f))
    print(f"{r['run']}: test RSE {r['test_rse']:.4f}, CORR {r['test_corr']:.4f}, best epoch {r['best_epoch']}")
shutil.copytree("results/runs", f"{CKPT}/results_runs", dirs_exist_ok=True)
print("copied to Google Drive:", f"{CKPT}/results_runs")

Then download `metrics.json` from Drive into `results/runs/<run name>/` in your local repo and
commit it. The `test_predictions.npy` files stay on Drive (needed later for the
Diebold–Mariano test, too big for git).